# ASSIGNMENT 3
Topics Covered: File & Exception Handling, NumPy Foundations and Matplotlib Basics,
Pandas Essentials and Seaborn, Pandas Advanced and Plotly.



Question 1: Explain the difference between:

● "r", "w", "a", "rb"

● Why context managers (with open()) are preferred

● What happens if a file is not properly closed in a production system

Ans 1.
### (i). Differences Between File Modes

* *"r" (Read):* It opens a file for reading text. The file pointer is placed at the beginning of the file. If the file does not exist, it raises a FileNotFoundError.
* *"w" (Write):* It opens a file for writing text. If the file already exists, it truncates (overwrites) all existing content. If the file does not exist, it creates a new one.
* *"a" (Append):* It opens a file for writing text, but places the file pointer at the very end of the file. Existing data is preserved, and any new data is appended to the end. If the file does not exist, it creates a new one.
* *"rb" (Read Binary):* It opens a file for reading data in binary mode (returning bytes rather than a decoded str). It does not perform newline translation and is used for non-text files like images, audio, or compiled binaries. Like "r", it raises FileNotFoundError if the file doesn't exist.

---

### (ii). Why Context Managers (with open()) Are Preferred

* *Deterministic Resource Cleanup:* The context manager guarantees that the file handle is closed as soon as execution leaves the with block via its __exit__() method.
* *Exception Safety:* Even if an unhandled error, crash, or early return/break occurs inside the block, the file is guaranteed to close properly without requiring boilerplate try...finally blocks.
* *Readability and Scope:* It clearly delineates the scope in which the file resource is active, preventing accidental reads or writes later in the code.

---

### (iii). What Happens If a File Is Not Properly Closed in a Production System

* *File Descriptor Exhaustion:* Operating systems enforce a strict per-process limit on open file descriptors (e.g., ulimit -n on Linux). Leaking open handles eventually triggers an OSError: [Errno 24] Too many open files, causing database connections, network sockets, and new file operations to fail entirely.
* *Data Loss or Corruption (Buffered I/O):* Standard write operations write to an in-memory buffer before flushing to disk. If a file is not closed and the process terminates abnormally, unwritten buffer data is lost.
* *File Locking Issues:* On platforms like Windows (and specific networked filesystems like NFS/SMB), an unclosed file maintains an active lock, preventing other processes or threads from reading, moving, modifying, or deleting the file.
* *Memory Leaks:* Open file handles retain system-level tracking structures and Python wrapper objects in memory, increasing the memory footprint of long-running services.


*****************************************************************


Question 2: You are given a CSV file sales.csv containing:

product,price,quantity

Laptop,50000,5

Phone,20000,10

Tablet,15000,7




Tasks:

1. Read the file using Python.
2. Compute total revenue per product.
3. Write results to a new file revenue.csv.
4. Explain limitations of manual CSV handling vs Pandas.

Ans 2 :
### Solution for Tasks 1, 2, and 3

#### Option A: Using Python’s Built-in csv Module


In [ ]:
import csv

input_file = "sales.csv"
output_file = "revenue.csv"

# 1. Read the input file and compute total revenue per product
processed_data = []

with open(input_file, mode="r", newline="", encoding="utf-8") as infile:
    reader = csv.DictReader(infile)
    for row in reader:
        product = row["product"]
        price = float(row["price"])
        quantity = int(row["quantity"])
        total_revenue = price * quantity

        processed_data.append(
            {
                "product": product,
                "price": price,
                "quantity": quantity,
                "total_revenue": total_revenue,
            }
        )

# 2. Write the calculated results to revenue.csv
with open(output_file, mode="w", newline="", encoding="utf-8") as outfile:
    fieldnames = ["product", "price", "quantity", "total_revenue"]
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(processed_data)

print(f"Results successfully saved to {output_file}")

#### Option B: Using pandas

In [ ]:
import pandas as pd

df = pd.read_csv("sales.csv")
df["revenue"] = df["price"] * df["quantity"]
df[["product", "revenue"]].to_csv("revenue.csv", index=False)


### Task 4: Limitations of Manual CSV Handling vs. Pandas

Limitations of Manual CSV Handling vs. Pandas

(i) Type Handling: Manual CSV treats values as strings and requires manual conversion. Pandas automatically detects data types.

(ii) Performance: Manual loops are slower for large datasets, while Pandas uses fast vectorized operations.

(iii) CSV Complexity: Manual splitting can fail with commas, quotes, and multi-line fields. Pandas handles these automatically.

(iii) Missing Data: Manual handling requires many checks, while Pandas provides isna(), fillna(), and dropna().

(iv) Data Analysis: Operations like grouping, merging, and reshaping require more code manually but are simple in Pandas.

(v) Large Files: Pandas supports chunking and memory-efficient data types for handling large datasets.

Question 3:  You receive JSON data from an API:

    {

     "customers": [

        {"id": 1, "name": "Amit", "purchase": 5000},

        {"id": 2, "name": "Sara", "purchase": 7000}

     ]
    }


Tasks:
1. Load the JSON.
2. Extract all customer names.
3. Compute total purchase value.
4. Explain why JSON is widely used in ML pipelines.

In [ ]:
#Ans 3 python
import json

# Input JSON string received from the API
api_response = """
{
  "customers": [
    {"id": 1, "name": "Amit", "purchase": 5000},
    {"id": 2, "name": "Sara", "purchase": 7000}
  ]
}
"""

# 1. Load the JSON
data = json.loads(api_response)

# 2. Extract all customer names
customer_names = [customer["name"] for customer in data["customers"]]
print("Customer Names:", customer_names)  # Output: ['Amit', 'Sara']

# 3. Compute total purchase value
total_purchase = sum(customer["purchase"] for customer in data["customers"])
print("Total Purchase Value:", total_purchase)  # Output: 12000





4. Why JSON is Widely Used in ML Pipelines
Ans: JSON is Widely Used in ML Pipelines because of the following:-

(i) Language-Agnostic Interoperability:* ML systems often span multiple languages (e.g., Python for training, C++/Go for high-throughput serving, JavaScript/Swift for frontends). JSON provides a standard text-based format natively parsed across virtually all environments.

(ii) Flexible Schema for Semi-Structured Data:* ML inputs often contain nested metadata, variable-length features, or optional parameters. JSON's key-value and array hierarchy handles sparse or nested data cleanly without rigid database schemas.

(iii)Standard for REST APIs & Microservices:* Serving models via tools like FastAPI, Flask, Triton, or TorchServe relies primarily on HTTP REST endpoints, where JSON is the standard serialization format for request payloads and model predictions.

(iv)Human-Readable & Easy to Debug:* Unlike binary protocols, JSON allows engineers to quickly inspect logged feature inputs, ground-truth labels, and model outputs during pipeline debugging and monitoring.


*******************************************************************

Question 4: Create NumPy arrays for:

● Product prices

● Quantities

Then:

1. Compute revenue using vectorized operations.
2. Compare performance of Python loop vs NumPy.
3. Explain broadcasting with example.

In [ ]:
#Ans 4 python
import time
import numpy as np

#  Create NumPy arrays for product prices and quantities[cite: 2]
prices = np.array([19.99, 45.50, 5.25, 120.00, 8.75])
quantities = np.array([10, 4, 50, 2, 25])

# -------------------------------------------------------------
# 1. Compute revenue using vectorized operations[cite: 2]
# -------------------------------------------------------------
# Element-wise product gives individual revenues, .sum() gives total
item_revenues = prices * quantities
total_revenue = np.sum(item_revenues)  # Alternatively: np.dot(prices, quantities)

print("Item Revenues:", item_revenues)
print(f"Total Revenue: Rs.{total_revenue:.2f}")

# -------------------------------------------------------------
# 2. Compare performance of Python loop vs NumPy[cite: 2]
# -------------------------------------------------------------
# Benchmark on a larger dataset (1,000,000 items)
n = 1_000_000
large_prices_np = np.random.uniform(5.0, 100.0, size=n)
large_quantities_np = np.random.randint(1, 50, size=n)

large_prices_list = large_prices_np.tolist()
large_quantities_list = large_quantities_np.tolist()

# Pure Python loop
start_py = time.perf_counter()
py_total = sum(p * q for p, q in zip(large_prices_list, large_quantities_list))
end_py = time.perf_counter()
py_time = end_py - start_py

# NumPy vectorized operation
start_np = time.perf_counter()
np_total = np.dot(large_prices_np, large_quantities_np)
end_np = time.perf_counter()
np_time = end_np - start_np

print(f"\n--- Performance Comparison (n={n:,}) ---")
print(f"Python Loop Time : {py_time:.5f} s")
print(f"NumPy Vectorized : {np_time:.5f} s")
print(f"Speedup          : {py_time / np_time:.1f}x faster")




'''
### 3. Explanation of Broadcasting with Example
Broadcasting is NumPy's mechanism that allows arithmetic operations between arrays of different shapes without copying data in memory.
The smaller array is conceptually "stretched" or repeated across the dimensions of the larger array to make their shapes compatible.

Broadcasting Rules

Two dimensions are compatible when:

1. They are equal in size, or
2. One of the dimensions is $1$.

#### Example: Applying a Uniform Discount

Applying a scalar discount or a per-category tax across multiple rows uses broadcasting automatically:

python
'''
import numpy as np

# Array of product prices: shape (4,)
prices = np.array([100.0, 200.0, 50.0, 300.0])

# Scalar discount (10% off): shape ()
discount_factor = 0.90

# The scalar is broadcast across every element in 'prices'
discounted_prices = prices * discount_factor
print("Discounted Prices:", discounted_prices)
# Output: [ 90. 180.  45. 270.]

# Matrix Example: Adjusting a (3, 2) price table by a (1, 2) store adjustment
store_prices = np.array([[10, 20], [30, 40], [50, 60]])  # Shape (3, 2)
tax_rates = np.array([1.05, 1.10])  # Shape (2,) -> broadcast to (3, 2)

final_prices = store_prices * tax_rates
print("\nFinal Prices with Tax:\n", final_prices)


Question 5:  Using sales data:
1. Plot line chart for monthly revenue.
2. Plot bar chart for product comparison.
3. Add labels, title, legend.
4. Explain why visualization is essential before modeling.

In [ ]:
# Ans 5 python
import matplotlib.pyplot as plt
import pandas as pd

# Sample sales data
monthly_data = pd.DataFrame(
    {
        "Month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun"],
        "Revenue": [12000, 15000, 14000, 19000, 22000, 26000],
    }
)

product_data = pd.DataFrame(
    {
        "Product": ["Laptop", "Headphones", "Monitor", "Keyboard", "Mouse"],
        "Units_Sold": [120, 450, 210, 320, 500],
    }
)

# Set up figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Line Chart: Monthly Revenue
axes[0].plot(
    monthly_data["Month"],
    monthly_data["Revenue"],
    marker="o",
    color="#1f77b4",
    linewidth=2,
    label="Monthly Revenue",
)
axes[0].set_title("Monthly Revenue Trend (H1)")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Revenue ($)")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

# 2. Bar Chart: Product Sales Comparison
axes[1].bar(
    product_data["Product"],
    product_data["Units_Sold"],
    color="#2ca02c",
    label="Units Sold",
)
axes[1].set_title("Product Sales Comparison")
axes[1].set_xlabel("Product Category")
axes[1].set_ylabel("Units Sold")
axes[1].grid(axis="y", linestyle="--", alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()




'''
# (4) Why Visualization is Essential Before Modeling
Ans: Visualization is essential before modeling because of the following:-
(i)Catches Hidden Patterns:** Numbers like mean and standard deviation can deceive (e.g., Anscombe’s Quartet); plots show the real shape of the data.
(ii)Spots Outliers Early:** Instantly highlights extreme values, entry errors (like age = 999), and noise that would distort model training.
(iii)Checks Assumptions:** Reveals skewness, non-normality, or multiple peaks, signaling if transformations (like log-scaling) are needed.
(iv)Shows Real Relationships:** Identifies whether variables are linear, curved, or clustered—relationships a single correlation score often misses.
(v)Flags Redundant Features:** Heatmaps quickly detect multicollinearity (features carrying duplicate information).
(vi)Guides Model Choice:** Tells you whether to use a simple linear model, a tree-based ensemble, or a neural network based on class separability and complexity.
'''

Question 6: Load a CSV into Pandas and:
1. Display first 5 rows.
2. Check missing values.
3. Compute summary statistics.
4. Select rows where sales > 15000.

In [ ]:
%%writefile sales_data.csv
product,category,sales,region
Laptop,Electronics,25000,East
Phone,Electronics,18000,West
Tablet,Electronics,10000,North
Keyboard,Accessories,5000,South
Mouse,Accessories,3000,East
Monitor,Electronics,22000,West
Headphones,Accessories,12000,North

In [ ]:
# Ans 6 python
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv("sales_data.csv")

# 1. Display the first 5 rows
print("--- First 5 Rows ---")
print(df.head(5))

# 2. Check for missing values in each column
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

# 3. Compute summary statistics (mean, std, min, percentiles, max)
print("\n--- Summary Statistics ---")
print(df.describe())

# 4. Select rows where sales > 15000
high_sales = df[df["sales"] > 15000]

print(f"\n--- Records with Sales > 15000 (Count: {len(high_sales)}) ---")
print(high_sales.head())


Question 7: Dataset contains:

● Missing values

● Duplicate rows

● Inconsistent capitalization

Tasks:
1. Remove duplicates.
2. Handle missing values.
3. Standardize text columns.
4. Explain impact of dirty data on ML performance.

In [ ]:
#Ans 7 python
import numpy as np
import pandas as pd

# Example dirty DataFrame
df = pd.DataFrame(
    {
        "customer_id": [101, 102, 103, 103, 104, 105],
        "category": [
            "Electronics",
            "electronics ",
            "HOME",
            "HOME",
            "  home",
            np.nan,
        ],
        "city": [
            "New York",
            "new york",
            "Chicago",
            "Chicago",
            "LOS ANGELES",
            "Chicago",
        ],
        "sales": [250.0, np.nan, 150.0, 150.0, 300.0, 180.0],
    }
)

# -------------------------------------------------------------------------
# 1. Remove duplicates
# -------------------------------------------------------------------------
df_cleaned = df.drop_duplicates().copy()

# -------------------------------------------------------------------------
# 2. Handle missing values
# -------------------------------------------------------------------------
# Strategy: Impute numeric features with median, categorical with mode (or a placeholder)
df_cleaned["sales"] = df_cleaned["sales"].fillna(df_cleaned["sales"].median())
df_cleaned["category"] = df_cleaned["category"].fillna(
    df_cleaned["category"].mode()[0]
)

# -------------------------------------------------------------------------
# 3. Standardize text columns
# -------------------------------------------------------------------------
text_columns = ["category", "city"]
for col in text_columns:
    # Strip leading/trailing whitespaces and convert to uniform lower case
    df_cleaned[col] = df_cleaned[col].str.strip().str.lower()

print("Cleaned Dataset:")
print(df_cleaned)

#--------------------------------------------------------------------------
'''

4. Impact of Dirty Data on ML Performance
Ans: Dirty data can significantly impact ML performance are as follows:-
(i)Artificially Inflated Cardinality and Sparse Encodings:* Inconsistent casing and untrimmed whitespace (e.g., "Home", "home", " home ") cause categorical encoders (like One-Hot or Target Encoding) to treat identical categories as distinct entities. This unnecessarily increases dimensionality, leads to sparse matrix representation, and degrades generalization on unseen data.
(ii)Biased Parameter Estimation and Loss Calculation:* Missing values that are improperly handled (or dropped arbitrarily) distort underlying statistical distributions. Dropping rows with missing values introduces sampling bias, while improper imputation can artificially reduce variance and mislead gradient-based optimization algorithms.
(iii)Data Leakage and Overfitting via Duplicates:* Duplicate rows spread across training and validation/test splits cause data leakage. The model effectively memorizes training instances rather than learning generalized decision boundaries, resulting in deceptively high evaluation metrics that collapse in production.
(iv)Degraded Decision Boundaries and Convergence:* Noisy, inconsistent labels and features add stochastic noise to the loss landscape. This forces optimization algorithms (such as SGD or Adam) into sub-optimal local minima, slows down convergence rates, and elevates both training error and variance.
'''
#---------------------------------------------------------------------------

Question 8: Using Plotly:
1. Create interactive bar chart of sales.
2. Create pie chart for category distribution.
3. Add hover information.
4. Explain difference between:

○ Static visualization (Matplotlib/Seaborn)

○ Interactive dashboards (Plotly)

Discuss where interactive dashboards are critical in business environments.

In [ ]:
#Ans 8python
import pandas as pd
import plotly.express as px

# Sample business dataset
data = {
    "Product": ["Laptops", "Phones", "Monitors", "Accessories", "Chairs"],
    "Category": [
        "Electronics",
        "Electronics",
        "Electronics",
        "Peripherals",
        "Furniture",
    ],
    "Sales": [120000, 95000, 45000, 28000, 35000],
    "Profit_Margin": ["18%", "22%", "15%", "40%", "25%"],
    "Units_Sold": [120, 190, 150, 700, 140],
}
df = pd.DataFrame(data)

# -------------------------------------------------------------
# 1. Interactive Bar Chart of Sales (with custom hover data)
# -------------------------------------------------------------
bar_fig = px.bar(
    df,
    x="Product",
    y="Sales",
    color="Category",
    title="Product Sales Breakdown",
    # 3. Add hover information
    hover_data={"Units_Sold": True, "Profit_Margin": True, "Sales": ":$,.2f"},
    labels={"Sales": "Total Revenue ($)", "Product": "Product Name"},
    template="plotly_white",
)
bar_fig.show()

# -------------------------------------------------------------
# 2. Pie Chart for Category Distribution (with custom hover data)
# -------------------------------------------------------------
pie_fig = px.pie(
    df,
    names="Category",
    values="Sales",
    title="Sales Distribution by Category",
    # 3. Add hover information
    hover_data={"Sales": ":$,.2f"},
    hole=0.3,  # Donut style for modern readability
)
pie_fig.update_traces(
    textinfo="percent+label",
    hovertemplate="<b>Category:</b> %{label}<br><b>Sales:</b> $%{value:,.2f}<br><b>Share:</b> %{percent}",
)
pie_fig.show()



#------------------------------------------------------------------------------------------------
'''
### 4. Static Visualization vs. Interactive Dashboards
Static Visualization:It is a fixed representation of data, such as a bar chart, pie chart, line graph, or infographic.
The information and design remain unchanged, so users can only view and interpret the data presented.
It is simple, easy to create, and useful for reports, textbooks, presentations, and printed documents.

Interactive Visualization: It allows users to actively explore data using features such as clicking, hovering, zooming, filtering, and selecting options.
The displayed information can change according to the user's actions.
It is useful for large or complex datasets because users can focus on specific details and discover patterns more easily.


#### ### Where Interactive Dashboards Are Critical in Business Environments

Interactive dashboards are critical in business environments because they help organizations monitor performance, analyze data, and make faster decisions.
Managers can view important information such as sales, revenue, expenses, customer behavior, and employee performance in real time.
They can also filter data by date, product, region, or department to understand specific business situations.

They are especially useful in areas such as sales and marketing, finance, inventory management, human resources, and customer service.
For example, a sales dashboard can show which products are selling the most, while an inventory dashboard can identify low-stock items.
By providing clear and interactive visual information, dashboards help businesses identify trends, detect problems, track targets, and improve decision-making.
'''


Question 9:
Write a Python program to:
1. Create a text file called data.txt
2. Write 5 lines of product information
3. Read the file and print only lines containing the word "Laptop"
4. Use a context manager
Explain why context managers prevent memory leaks.

In [ ]:
#Ans 9
# 1. Create a text file called data.txt and write 5 lines using a context manager
product_records = [
    "ID: 101, Product: Gaming Laptop, Price: Rs.120000, Stock: 15\n",
    "ID: 102, Product: Wireless Mouse, Price: Rs.250, Stock: 80\n",
    "ID: 103, Product: Ultrabook Laptop, Price: Rs.195000, Stock: 22\n",
    "ID: 104, Product: Mechanical Keyboard, Price: Rs.1750, Stock: 45\n",
    "ID: 105, Product: Budget Laptop, Price: Rs.45000, Stock: 30\n",
]

# Writing to the file using a context manager ('with' statement)
with open("data.txt", "w", encoding="utf-8") as file:
    file.writelines(product_records)

# 3. Read the file and print only lines containing the word "Laptop"
print("Lines containing 'Laptop':")
with open("data.txt", "r", encoding="utf-8") as file:
    for line in file:
        if "Laptop" in line:
            print(line.strip())



#-----------------------------------------------------------
'''
### Why Context Managers Prevent Memory Leaks and Resource Leaks
Ans: Context Managers Prevent Memory Leaks and Resource Leaks because of the following reasons:-
(i) Automatic cleanup: Resources are automatically released after use.
(ii) Handles errors safely:Cleanup happens even if an exception occurs.
(iii) Closes resources:Files, database connections, and network connections are properly closed.
(iv) Prevents resource leaks:Avoids keeping unused resources occupied.
(v) Cleaner code: The `with` statement reduces the need for manual cleanup.
(vi) Improves reliability: Makes programs safer and more efficient.

'''


Question 10: You are hired as a Data Analyst at an online learning company.


The company offers multiple courses and wants to understand:

● Revenue trends

● Student performance

● Course popularity

● Engagement patterns

You must design a simple analytics solution using Python and Data Science tools.

Design a CSV file structure that stores:

● Course Name

● Student ID

● Course Fee

● Completion Percentage

● Rating (1–5)

Write a Python program to:

1. Create this CSV file.

2. Add at least 10 sample records.

3. Read the file and calculate total revenue.

Explain why using with open() is better than manually opening and closing files.

In [ ]:
#Ans 10 python
import csv

# -------------------------------------------------------------
# 1. Design CSV structure & 2. Write at least 10 sample records
# -------------------------------------------------------------
csv_filename = "course_analytics.csv"

# Columns: Course Name, Student ID, Course Fee, Completion Percentage, Rating (1–5)
header = ["Course Name", "Student ID", "Course Fee", "Completion Percentage", "Rating"]

sample_records = [
    ["Python for Data Science", "STU_1001", 120.00, 95.0, 5],
    ["Machine Learning A-Z", "STU_1002", 200.00, 80.5, 4],
    ["Deep Learning Specialization", "STU_1003", 250.00, 60.0, 4],
    ["SQL for Analysts", "STU_1004", 80.00, 100.0, 5],
    ["Python for Data Science", "STU_1005", 120.00, 40.0, 3],
    ["Data Visualization with Plotly", "STU_1006", 90.00, 75.0, 4],
    ["Machine Learning A-Z", "STU_1007", 200.00, 100.0, 5],
    ["Cloud Data Engineering", "STU_1008", 300.00, 25.0, 2],
    ["SQL for Analysts", "STU_1009", 80.00, 90.0, 4],
    ["Python for Data Science", "STU_1010", 120.00, 85.0, 5],
    ["Cloud Data Engineering", "STU_1011", 300.00, 70.0, 4],
]

# Write records to CSV using a context manager
with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(header)
    writer.writerows(sample_records)

print(f"File '{csv_filename}' created successfully with {len(sample_records)} records.\n")

# -------------------------------------------------------------
# 3. Read the file and calculate total revenue
# -------------------------------------------------------------
total_revenue = 0.0

with open(csv_filename, mode="r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        total_revenue += float(row["Course Fee"])

print(f"Total Revenue: ${total_revenue:,.2f}")



#----------------------------------------------
''''
### ### Why `with open()` Is Better

(i) Automatic closing: The file closes automatically after use.
(ii) Prevents resource leaks: Avoids leaving files open unnecessarily.
(iii) Exception-safe: The file is closed even if an error occurs.
(iv) Less code: No need to write `file.close()` manually.
(v) Easy to read: Makes file-handling code simple and clear.
(vi) Safer:Reduces the chance of forgetting to close the file.

'''